In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score

import warnings
warnings.filterwarnings("ignore")

/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
batch_size = 16
epochs = 3
learning_rate = 2e-5

In [3]:
model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***loading conv info***

In [4]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)

***loading llama labled messages***

In [5]:
messages_with_lables_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic_fixed.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

***loading llama labled conversation chunks***

In [7]:
conv_chunks_with_lables_path = 'trainDatasets/message_chunks_based_on_cutoff.csv'

conv_chunks_with_lables_df = pd.read_csv(conv_chunks_with_lables_path)

conv_chunks_with_lables_df['engagement_id'] = conv_chunks_with_lables_df['engagement_id'].astype(str)
conv_chunks_with_lables_df = conv_chunks_with_lables_df[conv_chunks_with_lables_df['anonymized'].notna()]
conv_chunks_with_lables_df['name'] = conv_chunks_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_conv_chunks_df = conv_chunks_with_lables_df[conv_chunks_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_conv_chunks_df = conv_chunks_with_lables_df[conv_chunks_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

***loading all messages***

In [8]:
all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df = all_messages_df[all_messages_df["seeker"]]

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()


## creating Dataset objects for train and test

In [9]:
# mapping the text into inputs that fits the model
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

def make_weighted_train_loaders(dfs, weights, tokenize):
    #make sampled df
    acc_train = pd.DataFrame(data= {})
    for df, weight in zip(dfs,weights):
        sample = df.sample(frac = weight)
        acc_train = pd.concat([acc_train,sample[["anonymized" , "label"]]])
    
    #create datasets
    train_dataset = Dataset.from_pandas(acc_train)
    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=16)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    return train_loader

def make_test_loader(df, tokenize):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(tokenize, batched=True, batch_size=16)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return test_loader

In [10]:
#labled_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [1,0], tokenize)
#all_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [0,1], tokenize)
labled_messages_test_loader = make_test_loader(test_labled_messages_df, tokenize)
conv_chunks_test_loader = make_test_loader(test_conv_chunks_df, tokenize)
all_messages_test_loader = make_test_loader(test_all_messages_df, tokenize)

Map: 100%|██████████| 6047/6047 [00:03<00:00, 1558.97 examples/s]


## train model

***train on labled messages***

In [11]:
def general_trainer(train_loader, loss_fn=None):
    optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
    bert_model.train()
    
    total_batches = len(train_loader)
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)
        if loss_fn is None:
            loss = outputs.loss
        else:
            loss = loss_fn(outputs, labels)
        
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")

In [ ]:
plan = [
    [0.9, 0.1, 0.02],
    [0.8, 0.2, 0.05],
    [0.75, 0.3, 0.1],
    
    [0.4, 0.8, 0.2],
    [0.3, 0.9, 0.3],
    [0.2, 0.8, 0.4],
    
    [0.1, 0.2, 0.8],
    [0.05, 0.1, 0.9],
    [0.02, 0.05, 0.95]
]
for w in plan:
    train_loader = make_weighted_train_loaders([train_labled_messages_df, train_conv_chunks_df, train_all_messages_df] , w, tokenize)
    general_trainer(train_loader)

Map: 100%|██████████| 308598/308598 [01:15<00:00, 4090.56 examples/s]


Batch 1000/19288 completed. 18288 remaining.
Batch 2000/19288 completed. 17288 remaining.
Batch 3000/19288 completed. 16288 remaining.
Batch 4000/19288 completed. 15288 remaining.
Batch 5000/19288 completed. 14288 remaining.
Batch 6000/19288 completed. 13288 remaining.
Batch 7000/19288 completed. 12288 remaining.
Batch 8000/19288 completed. 11288 remaining.


## eval model

In [12]:
def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1: {f1}')
    print(f'ROC-AUC: {roc_auc}')
    print(f'F2: {f2}')

***eval messages***

In [13]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in labled_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.9514147342303655
Precision: 0.7783751493428913
Recall: 0.7460208404900951
F1: 0.7618546453838508
ROC-AUC: 0.9733613769600454
F2: 0.7522747217218604


***eval chunks***

In [14]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in conv_chunks_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.7908410696965498
Precision: 0.8910037416625997
Recall: 0.699846664962944
F1: 0.7839404565948616
ROC-AUC: 0.9050186652766683
F2: 0.7312221302501936


***eval conv***

In [15]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in all_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.869852819579957
Precision: 0.617199391171994
Recall: 0.7406392694063927
F1: 0.6733084267330842
ROC-AUC: 0.9050007007915255
F2: 0.7121531436599929


In [16]:
name_for_save = input("enter model name: ")
torch.save(bert_model.state_dict(), f"model_weights_{name_for_save}.pth")

enter model name:  progressive_curiculum_learning_conv_chunks_messages_llama_generated
